# IOAI — 2025 Selection Camp Skeleton Action (Colab 자동 설정판)

아래 **설정 셀을 먼저 실행**하면 공개 데이터 소스에서 데이터를 받아 이 폴더에 `train.csv`/`test.csv` 등으로 준비합니다. 이후 셀이 그대로 학습/예측하고, 만들어진 제출 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

> 런타임 메뉴 → **런타임 유형 변경 → GPU** (필요 시).

In [ ]:
# === 데이터 자동 준비 (가장 먼저 실행) ===
import os, urllib.request
os.makedirs('data/seq', exist_ok=True)
_base = 'https://raw.githubusercontent.com/scvcoder/ioai-colab/main/data/skeletons-seq'
for f in ['train.pt', 'val.pt', 'test.pt']:
    if not os.path.exists(f'data/seq/{f}'):
        urllib.request.urlretrieve(f'{_base}/{f}', f'data/seq/{f}')
print('데이터 준비:', 'data/seq/train.pt' if os.path.exists('data/seq/train.pt') else '실패')
import os; print('작업 폴더:', os.getcwd()); print('내용:', sorted(os.listdir('.')))

# Skeletons — 골격 시퀀스 행동·카메라 분류 (베이스라인)

> 데이터는 DGX에 **사전 준비**돼 있습니다 — `data/seq/{train,val,test}.pt`(IDSample별 가변길이 골격 시퀀스 `[T,75]`(25관절×XYZ) + train/val 의 Action·Camera 라벨). 느린 CSV 그룹화 없이 바로 학습합니다.

**과제**: 골격 관절좌표 시퀀스로 **Action(5클래스)**·**Camera(3클래스)** 를 분류. 지표 = 두 정확도 평균 **mean_acc**(높을수록 좋음). held-out val(IDSample%5==0)로 채점, 제출 `submission.csv`(datapointID, action, camera).

베이스라인: 시퀀스를 **프레임축 통계(관절별 평균·표준편차=150차원)** 로 요약 → 라벨별 로지스틱 회귀. 정적 통계라 **Action 은 잘 맞지만 Camera(시점, 시간 동역학 필요)는 약합니다**. (개선: **LSTM** 시퀀스 모델 → 모범답안)

In [ ]:
import torch, numpy as np, pandas as pd
from sklearn.linear_model import LogisticRegression
tr = torch.load('data/seq/train.pt'); va = torch.load('data/seq/val.pt')
def feat(seqs): return np.stack([np.concatenate([s.mean(0).numpy(), s.std(0).numpy()]) for s in seqs])
Xtr, Xva = feat(tr['seqs']), feat(va['seqs'])
print('train', Xtr.shape, '| val', Xva.shape)

In [ ]:
pred = {}
for lab in ['action','camera']:
    clf = LogisticRegression(max_iter=1000).fit(Xtr, tr[lab]); pred[lab] = clf.predict(Xva)
    print(lab, 'acc', round(float((pred[lab]==np.array(va[lab])).mean()),4))
pd.DataFrame({'datapointID': va['ids'], 'action': pred['action'], 'camera': pred['camera']}).to_csv('submission.csv', index=False)
print('wrote submission.csv', len(va['ids']))

## 제출 파일 모으기
아래 셀을 실행하면 제출 파일이 **최상위(`/content`)로 복사**되어 왼쪽 파일 탐색기에 바로 보입니다.
그 파일을 내려받아 연습 사이트 **Submissions** 탭에 올리면 채점됩니다.

In [ ]:
# === 제출 파일을 /content 로 모으기 (마지막에 실행) ===
import os, glob, shutil
TARGETS = ['submission.csv']
OUT = "/content" if os.path.isdir("/content") else os.getcwd()
found = []
for name in TARGETS:
    hits = [name] if os.path.exists(name) else glob.glob(f"**/{name}", recursive=True)
    if not hits:
        print("아직 없음(해당 셀을 먼저 실행하세요):", name); continue
    dst = os.path.join(OUT, os.path.basename(hits[0]))
    if os.path.abspath(hits[0]) != os.path.abspath(dst):
        shutil.copy2(hits[0], dst)
    found.append(dst)
print("제출 파일 저장 위치(파일 탐색기 최상위):", found)